# Parte 2 (continuacion) - Denoising con preentrenamiento
Hecho por Samuel Patiño y Nicolás Peña - IA Generativa UAO

## Por que existe este cuaderno

En el cuaderno 02 corrimos ocho estrategias con dos niveles de ruido y el resultado fue claro:

| Estrategia | PSNR (ruido 0.1) | SSIM |
|---|---|---|
| base_profe | 16.69 | 0.5714 |
| skip_concat | 16.68 | 0.5827 |
| mas_profundo | 16.78 | 0.5691 |
| sin_compresion | 21.58 | 0.6631 |
| residual | 20.54 | 0.6165 |
| **parches** | **25.16** | **0.8190** |

La unica estrategia que dio un salto grande fue `parches`, que es la unica que **aumenta la
cantidad de datos**. Todo lo demas (cambiar la arquitectura, quitar el pooling, usar salida
residual) movio la aguja muy poco o la empeoro.

La conclusion es que el factor limitante no es el diseno de la red sino la cantidad de datos:
tenemos 200 imagenes de entrenamiento, frente a las 50.000 que usan los trabajos de referencia.

## Que hacemos aca

Tres cosas, en orden de impacto:

1. **Preentrenar** el autoencoder con un dataset publico de 21.165 radiografias, que tiene las
   mismas clases que el nuestro. El denoising no necesita etiquetas, asi que podemos usar
   todas esas imagenes sin problema.
2. **Multiplicar los parches**: en el cuaderno 02 sacabamos 4 recortes por imagen. Aca sacamos
   81 recortes traslapados, pasando de 1.004 a unas 16.200 muestras.
3. **Aumento de datos**: ruido nuevo en cada epoca mas volteo horizontal aleatorio.

Despues comparamos el modelo preentrenado contra uno entrenado desde cero, para medir cuanto
aporto realmente el preentrenamiento.

En Kaggle: Settings > Accelerator > GPU y Internet ON.

In [ ]:
# revisamos GPU rapido
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# librerias basicas
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Conv2DTranspose,
                                     BatchNormalization, Dropout, LeakyReLU, Dense,
                                     Flatten, Reshape, Add)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import load_img, img_to_array
%matplotlib inline
np.random.seed(42)
tf.random.set_seed(42)

# hiperparametros
IMG = 64
BATCH = 32
LR = 0.001
LR_FT = 0.0005        # learning rate mas bajo para el ajuste fino
LATENT = 256
EPOCHS_PRE = 15       # epocas de preentrenamiento con el dataset externo
EPOCHS_FT = 20        # epocas de ajuste fino con nuestros datos
LIMITE_EXTERNO = 12000  # cuantas imagenes externas usamos
STRIDE = 8            # paso de los recortes traslapados

# nivel de ruido principal: 0.1 es el realista, 0.5 es el del enunciado
NIVEL = 0.1
NIVEL_EXAMEN = 0.5

## 1. Metricas

Las mismas del cuaderno 02. PSNR mas alto es mejor, SSIM mas alto es mejor, MSE mas bajo es
mejor. La referencia para interpretarlas:

- PSNR: menos de 20 dB malo, 20-30 aceptable, 30-40 bueno, mas de 40 excelente.
- SSIM: mayor a 0.90 muy bueno, entre 0.70 y 0.90 bueno, menor a 0.70 regular.

In [ ]:
def calcular_metricas(a, b):
    """a = imagen a evaluar, b = imagen limpia de referencia. Ambas en [0,1]"""
    a = tf.convert_to_tensor(a, dtype=tf.float32)
    b = tf.convert_to_tensor(b, dtype=tf.float32)
    psnr = tf.image.psnr(a, b, max_val=1.0).numpy().mean()
    ssim = tf.image.ssim(a, b, max_val=1.0).numpy().mean()
    mse  = np.mean((a.numpy() - b.numpy()) ** 2)
    return psnr, ssim, mse

def mostrar_metricas(a, b, nombre):
    psnr, ssim, mse = calcular_metricas(a, b)
    print(f"{nombre:16s}  PSNR: {psnr:6.2f} dB   SSIM: {ssim:.4f}   MSE: {mse:.5f}")
    return psnr, ssim, mse

# ruido gaussiano, el factor entra como parametro
def add_noise(x, noise_factor=0.1):
    return np.clip(x + noise_factor * np.random.randn(*x.shape), 0., 1.)

## 2. Dataset externo para preentrenar

Usamos el **COVID-19 Radiography Database** de Kaggle: 21.165 radiografias de torax con las
mismas clases que nuestro dataset (COVID, Normal, Neumonia viral, mas Opacidad pulmonar).

Por que podemos usarlo sin problema:

- El denoising es **no supervisado**, no necesita etiquetas. Solo necesitamos imagenes limpias
  de radiografias para que la red aprenda su estructura.
- La evaluacion final se hace sobre nuestro dataset asignado, asi que los numeros que
  reportamos siguen siendo sobre nuestros datos.

Cargamos hasta 12.000 imagenes para no saturar la memoria. El cargador recorre las carpetas
ignorando las mascaras de segmentacion que trae el dataset.

In [ ]:
import kagglehub

try:
    base_ext = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
    EXTERNO_OK = True
    print("dataset externo en:", base_ext)
except Exception as e:
    print("no se pudo descargar el dataset externo:", e)
    EXTERNO_OK = False

def cargar_imagenes(raiz, limite=None, size=64):
    """recorre carpetas y carga imagenes, ignorando las mascaras de segmentacion"""
    rutas = []
    for root, dirs, files in os.walk(raiz):
        if "mask" in root.lower():
            continue
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                rutas.append(os.path.join(root, f))
    np.random.shuffle(rutas)
    if limite:
        rutas = rutas[:limite]
    xs = []
    for p in rutas:
        try:
            img = load_img(p, target_size=(size, size), color_mode="rgb")
            xs.append(img_to_array(img) / 255.0)
        except Exception:
            pass
    return np.array(xs, dtype="float32")

if EXTERNO_OK:
    x_ext = cargar_imagenes(base_ext, limite=LIMITE_EXTERNO)
    print("imagenes externas cargadas:", x_ext.shape)

## 3. Arquitectura

Usamos la del profesor de la semana 5, sin modificaciones. Es la que gano en el cuaderno 02
cuando se combino con parches, asi que no tiene sentido cambiarla: lo que queremos medir aca
es el efecto de los datos, no el de la arquitectura.

Recordatorio del flujo: 64x64x3 baja a 32, 16 y 8, pasa por un latente denso de 256, y vuelve
a subir con convoluciones transpuestas hasta 64x64x3. La conexion skip suma la salida de 32x32
del encoder al decoder.

In [ ]:
def construir_ae(dropout=0.3):
    inputs = Input(shape=(IMG, IMG, 3))

    # encoder
    x = Conv2D(32, 3, activation="relu", padding="same")(inputs)   # 64x64x32
    x = BatchNormalization()(x)
    x = MaxPooling2D()(x)                                          # 32x32x32
    x = Dropout(dropout)(x)

    skip = Conv2D(32, 3, padding="same")(x)                        # 32x32x32
    x = LeakyReLU()(skip)
    x = BatchNormalization()(x)
    x = MaxPooling2D()(x)                                          # 16x16x32
    x = Dropout(dropout)(x)

    x = Conv2D(64, 3, activation="relu", padding="same")(x)        # 16x16x64
    x = BatchNormalization()(x)
    x = MaxPooling2D()(x)                                          # 8x8x64

    # espacio latente
    x = Flatten()(x)
    latent = Dense(LATENT, activation="relu", name="latent")(x)

    # decoder espejo
    x = Dense(8*8*64, activation="relu")(latent)
    x = Reshape((8, 8, 64))(x)
    x = Conv2DTranspose(64, 3, activation="relu", strides=(2,2), padding="same")(x)  # 16
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)
    x = Conv2DTranspose(32, 3, activation="relu", strides=(2,2), padding="same")(x)  # 32
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)
    x = Conv2DTranspose(32, 3, padding="same")(x)
    x = Add()([x, skip])
    x = LeakyReLU()(x)
    x = BatchNormalization()(x)
    decoded = Conv2DTranspose(3, 3, activation="sigmoid", strides=(2,2), padding="same")(x)

    return Model(inputs, decoded)

## 4. Preentrenamiento

Entrenamos la red con las imagenes externas: entrada ruidosa, salida limpia, perdida MSE. No
usamos etiquetas en ningun momento.

Este paso le enseña a la red como se ve una radiografia limpia antes de que vea nuestro
dataset. Es la misma idea del aprendizaje no supervisado que vimos en clase y que la Parte 3
del examen pide aplicar.

In [ ]:
modelo_pre = None
if EXTERNO_OK:
    tf.keras.backend.clear_session()
    np.random.seed(42)
    tf.random.set_seed(42)

    modelo_pre = construir_ae()
    modelo_pre.compile(optimizer=Adam(learning_rate=LR), loss="mse")

    hist_pre = modelo_pre.fit(
        add_noise(x_ext, NIVEL), x_ext,
        epochs=EPOCHS_PRE, batch_size=BATCH, shuffle=True,
        validation_split=0.1, verbose=1)

    plt.figure(figsize=(7,4))
    plt.plot(hist_pre.history["loss"], label="train")
    plt.plot(hist_pre.history["val_loss"], label="val")
    plt.xlabel("epocas"); plt.ylabel("mse")
    plt.title("Preentrenamiento con dataset externo")
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig("/kaggle/working/curva_preentrenamiento.pdf", format="pdf")
    plt.show()

## 5. Nuestro dataset asignado

Cargamos el dataset de COVID que nos toco, con la subcarpeta intermedia que ya conocemos.

In [ ]:
base = kagglehub.dataset_download("pranavraikokte/covid19-image-dataset")
DATA = os.path.join(base, "Covid19-dataset")
if not os.path.isdir(os.path.join(DATA, "train")):
    DATA = base
print("usando:", DATA)

def cargar_split(split, size=64):
    xs, ys = [], []
    clases = sorted(os.listdir(os.path.join(DATA, split)))
    mapa = {c: i for i, c in enumerate(clases)}
    for c in clases:
        carpeta = os.path.join(DATA, split, c)
        for f in os.listdir(carpeta):
            try:
                img = load_img(os.path.join(carpeta, f), target_size=(size, size),
                               color_mode="rgb")
                xs.append(img_to_array(img) / 255.0)
                ys.append(mapa[c])
            except Exception:
                pass
    return np.array(xs, dtype="float32"), np.array(ys), clases

x_train, y_train, clases = cargar_split("train")
x_test, y_test, _ = cargar_split("test")
print("train:", x_train.shape, " test:", x_test.shape, " clases:", clases)

## 6. Division y parches traslapados

Separamos 20 por ciento del train para validacion, manteniendo la proporcion de clases.

Despues viene la parte importante: **recortes traslapados**. En el cuaderno 02 cargabamos cada
imagen a 128x128 y sacabamos 4 recortes de 64x64 sin traslapar. Aca usamos un paso de 8 pixeles
en lugar de 64, asi que de cada imagen salen 81 recortes en vez de 4:

    ((128 - 64) / 8 + 1)^2 = 9 x 9 = 81 recortes por imagen

Con 200 imagenes de entrenamiento pasamos de 1.004 a unas 16.200 muestras.

Los recortes salen solamente de las imagenes de entrenamiento. Los de validacion y test no se
tocan, para que la evaluacion siga siendo honesta.

In [ ]:
# indices primero, para poder separar antes de recortar
idx = np.arange(len(x_train))
idx_tr, idx_val, _, _ = train_test_split(
    idx, y_train, test_size=0.2, random_state=42, stratify=y_train)

x_tr_base, x_val_base = x_train[idx_tr], x_train[idx_val]
print("entrenamiento:", x_tr_base.shape, " validacion:", x_val_base.shape)

# cargamos las de entrenamiento a 128 para poder recortar
x_tr_grande, _, _ = cargar_split("train", size=128)
x_tr_grande = x_tr_grande[idx_tr]
print("para recortar:", x_tr_grande.shape)

def extraer_parches(xs, size=128, patch=64, stride=8):
    """recortes traslapados: multiplica la cantidad de muestras"""
    out = []
    for im in xs:
        for i in range(0, size - patch + 1, stride):
            for j in range(0, size - patch + 1, stride):
                out.append(im[i:i+patch, j:j+patch])
    return np.array(out, dtype="float32")

x_tr_parches = extraer_parches(x_tr_grande, 128, 64, STRIDE)
print("parches de entrenamiento:", x_tr_parches.shape)

# ruido fijo de evaluacion, para que la comparacion sea justa
np.random.seed(0)
x_val_n = add_noise(x_val_base, NIVEL)
x_test_n = add_noise(x_test, NIVEL)
np.random.seed(42)

print()
print("--- punto de partida ---")
mostrar_metricas(x_val_n, x_val_base, "ruidosa")

## 7. Ajuste fino contra entrenamiento desde cero

Entrenamos dos modelos identicos con exactamente los mismos datos y las mismas epocas:

1. **preentrenado**: arranca desde los pesos del dataset externo y se ajusta con nuestros
   parches.
2. **desde cero**: arranca con pesos aleatorios y se entrena con los mismos parches.

La unica diferencia es de donde arrancan. Si el preentrenamiento sirve, el primero deberia
quedar claramente arriba.

En ambos casos usamos aumento de datos: **ruido nuevo en cada epoca** y **volteo horizontal
aleatorio**. El aumento se aplica primero a la imagen limpia y despues se le agrega el ruido,
asi la entrada y el objetivo quedan alineados. Si voltearamos despues de ensuciar, la imagen
ruidosa y la limpia ya no corresponderian.

In [ ]:
def lote_aumentado(x_limpio, nivel):
    """Voltea algunas imagenes limpias y despues las ensucia.

    Devuelve DOS cosas: la entrada ruidosa y el objetivo limpio, con el mismo
    aumento aplicado a ambas. Si voltearamos solo la entrada, el modelo tendria
    que reconstruir una imagen sin voltear a partir de una volteada, que es
    imposible, y el entrenamiento divergiria.
    """
    x = x_limpio
    if np.random.rand() < 0.5:
        x = x[:, :, ::-1, :]        # volteo horizontal, barato y sin interpolar
    return add_noise(x, nivel), x

def entrenar(modelo, nivel, epocas, lr, nombre):
    modelo.compile(optimizer=Adam(learning_rate=lr), loss="mse")

    # si la perdida de validacion se estanca, bajamos el learning rate
    reducir = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=0)

    hist = {"loss": [], "val_loss": []}
    mejor_val = float("inf")
    mejores_pesos = None

    for ep in range(epocas):
        xn, xc = lote_aumentado(x_tr_parches, nivel)
        h = modelo.fit(xn, xc, epochs=1, batch_size=BATCH, shuffle=True,
                       validation_data=(x_val_n, x_val_base), verbose=0,
                       callbacks=[reducir])
        hist["loss"] += h.history["loss"]
        hist["val_loss"] += h.history["val_loss"]

        # nos quedamos con los pesos del mejor momento, por si algo diverge al final
        v = h.history["val_loss"][0]
        if v < mejor_val:
            mejor_val = v
            mejores_pesos = [w.copy() for w in modelo.get_weights()]

        if (ep + 1) % 5 == 0:
            print(f"  {nombre} epoca {ep+1}/{epocas}  loss {h.history['loss'][0]:.5f}"
                  f"  val {h.history['val_loss'][0]:.5f}")

    if mejores_pesos is not None:
        modelo.set_weights(mejores_pesos)
        print(f"  {nombre}: se restauraron los pesos de la mejor epoca (val {mejor_val:.5f})")
    return hist

resultados = []
modelos = {}

# modelo 1: preentrenado y ajustado
if modelo_pre is not None:
    print("ajustando modelo preentrenado...")
    hist_ft = entrenar(modelo_pre, NIVEL, EPOCHS_FT, LR_FT, "preentrenado")
    pred = modelo_pre.predict(x_val_n, verbose=0)
    psnr, ssim, mse = calcular_metricas(pred, x_val_base)
    resultados.append(dict(nombre="preentrenado", psnr=psnr, ssim=ssim, mse=mse, hist=hist_ft))
    modelos["preentrenado"] = modelo_pre
    print(f"preentrenado   PSNR {psnr:6.2f} dB   SSIM {ssim:.4f}")

# modelo 2: desde cero
print()
print("entrenando desde cero...")
tf.keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)
modelo_cero = construir_ae()
hist_cero = entrenar(modelo_cero, NIVEL, EPOCHS_FT, LR, "desde cero")
pred = modelo_cero.predict(x_val_n, verbose=0)
psnr, ssim, mse = calcular_metricas(pred, x_val_base)
resultados.append(dict(nombre="desde cero", psnr=psnr, ssim=ssim, mse=mse, hist=hist_cero))
modelos["desde cero"] = modelo_cero
print(f"desde cero     PSNR {psnr:6.2f} dB   SSIM {ssim:.4f}")

## 8. Comparacion

Comparamos los dos modelos y tambien dejamos anotado el numero del cuaderno 02 como referencia,
que fue lo mejor que logramos antes de aplicar esta metodologia (25.16 dB con SSIM 0.8190,
usando solo 1.004 parches y sin preentrenamiento).

In [ ]:
# referencia del cuaderno 02, para ver cuanto mejoramos
REF_02 = dict(nombre="cuaderno 02", psnr=25.16, ssim=0.8190)

nombres = [r["nombre"] for r in resultados] + [REF_02["nombre"]]
psnrs = [r["psnr"] for r in resultados] + [REF_02["psnr"]]
ssims = [r["ssim"] for r in resultados] + [REF_02["ssim"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(nombres, psnrs, color=["steelblue", "darkorange", "grey"])
axes[0].set_title("PSNR (mas alto mejor)")
axes[0].set_ylabel("dB")
axes[0].grid(alpha=0.3, axis="y")
axes[1].bar(nombres, ssims, color=["steelblue", "darkorange", "grey"])
axes[1].set_title("SSIM (mas alto mejor)")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("/kaggle/working/comparacion_preentrenamiento.pdf", format="pdf")
plt.show()

print(f"{'modelo':16s} {'PSNR':>8s} {'SSIM':>8s}")
for n, p, s in zip(nombres, psnrs, ssims):
    print(f"{n:16s} {p:8.2f} {s:8.4f}")

# curvas de los dos modelos entrenados aca
plt.figure(figsize=(7,4))
for r in resultados:
    plt.plot(r["hist"]["val_loss"], label=r["nombre"])
plt.xlabel("epocas"); plt.ylabel("mse de validacion")
plt.title("Curvas de validacion")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/curvas_ajuste.pdf", format="pdf")
plt.show()

## 9. Evaluacion final sobre test

Elegimos el mejor modelo por PSNR de validacion y lo evaluamos sobre el conjunto de test, que
no se uso ni para entrenar ni para elegir. Este es el numero que va al informe.

In [ ]:
mejor = max(resultados, key=lambda r: r["psnr"])
modelo_final = modelos[mejor["nombre"]]
print("mejor modelo:", mejor["nombre"])

pred_test = modelo_final.predict(x_test_n, verbose=0)
print()
print("--- resultado final sobre test ---")
mostrar_metricas(x_test_n, x_test, "ruidosa")
mostrar_metricas(pred_test, x_test, "denoised")

## 10. Imagenes del modelo ganador

Tripleta original, ruidosa y reconstruida, y despues la grilla de 48 en el formato del
profesor.

In [ ]:
plt.figure(figsize=(15,6))
for i in range(10):
    ax = plt.subplot(3,10,i+1); plt.imshow(x_test[i]); plt.axis("off")
    if i==0: ax.set_title("original")
    ax = plt.subplot(3,10,i+11); plt.imshow(x_test_n[i]); plt.axis("off")
    if i==0: ax.set_title("ruidosa")
    ax = plt.subplot(3,10,i+21); plt.imshow(pred_test[i]); plt.axis("off")
    if i==0: ax.set_title("denoised")
plt.suptitle("Denoising COVID - " + mejor["nombre"])
plt.savefig("/kaggle/working/triplet_preentrenado.pdf", format="pdf")
plt.show()

n = 48
r = np.random.randint(0, len(x_test)-n)
imgs48, noisy48 = x_test[r:r+n], x_test_n[r:r+n]
den48 = modelo_final.predict(noisy48, verbose=0)
rows, cols = 4, 12
f = plt.figure(figsize=(cols*1.5, rows*1.5*3))
for i in range(rows):
    for j in range(cols):
        k = i*cols + j
        f.add_subplot(rows*3, cols, (i*cols)+(j+1))
        plt.imshow(imgs48[k]); plt.axis("off")
        f.add_subplot(rows*3, cols, (rows*cols)+(i*cols)+(j+1))
        plt.imshow(noisy48[k]); plt.axis("off")
        f.add_subplot(rows*3, cols, (2*rows*cols)+(i*cols)+(j+1))
        plt.imshow(den48[k]); plt.axis("off")
f.suptitle("Denoising COVID - original / ruidosa / denoised", fontsize=18)
plt.savefig("/kaggle/working/grid48_preentrenado.pdf", format="pdf")
plt.show()

## Conclusiones

Completar con los numeros reales despues de correr. Puntos para discutir:

1. **Cuanto aporto el preentrenamiento.** Comparar el modelo preentrenado contra el entrenado
   desde cero con los mismos datos. Si la diferencia es grande, queda demostrado que el
   problema era de cantidad de datos y no de arquitectura.

2. **Cuanto aportaron los parches traslapados.** Pasamos de 1.004 a unas 16.200 muestras.
   Comparar contra el 25.16 dB del cuaderno 02.

3. **Si llegamos a los numeros de la literatura.** El repositorio de anushkayadav reporta 24.83
   dB y SSIM 0.868 con un autoencoder simple sobre 50.000 imagenes. Si nos acercamos con 200
   imagenes mas preentrenamiento, es un resultado fuerte.

4. **Por que funciona.** La red aprende la variedad donde viven las radiografias limpias y
   proyecta cualquier punto corrupto de vuelta sobre ella. Cuantas mas radiografias vea durante
   el preentrenamiento, mejor aprende esa variedad.

5. **Honestidad sobre la limitacion.** Seguimos teniendo pocas imagenes propias. El
   preentrenamiento no las reemplaza, las complementa. Conviene decirlo.

6. **Transparencia metodologica.** Dejar claro en el informe que se preentreno con un dataset
   publico y que la evaluacion se hizo siempre sobre el dataset asignado.

## Referencias

1. Rahman, T. y otros (2021). COVID-19 Radiography Database. Kaggle.
   https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
   Dataset publico con 21.165 radiografias de torax. Se usa solamente para preentrenamiento no
   supervisado; la evaluacion se hace sobre el dataset asignado.

2. Vincent, P., Larochelle, H., Bengio, Y., Manzagol, P.-A. (2008). Extracting and Composing
   Robust Features with Denoising Autoencoders. Technical Report 1316, Universite de Montreal.
   Fundamento del denoising autoencoder y de la interpretacion por variedad (manifold).

3. Nishio, M. y otros (2017). Convolutional auto-encoder for image denoising of ultra-low-dose
   CT. Heliyon, 3(8), e00393.
   De aca sale la idea de entrenar con parches: ellos llegan a 100.000 pares a partir de pocas
   imagenes de tomografia.

4. Yadav, A. Denoising CIFAR-10. https://github.com/anushkayadav/Denoising_cifar10
   Referencia numerica: 24.83 dB y SSIM 0.868 con un autoencoder simple sobre 50.000 imagenes.

5. Paniagua, J. L. (2026). Semana 5: CNN Autoencoder con CIFAR-10. Universidad Autonoma de
   Occidente. Base de la arquitectura usada en este cuaderno.